In [124]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [125]:
df = pd.read_csv("diabetic_data.csv")

In [126]:
df = df.drop(columns=["encounter_id", "patient_nbr"], errors="ignore")

In [127]:
df.isna().sum(axis=0).sort_values(ascending=False).head(5)

max_glu_serum        96420
A1Cresult            84748
race                     0
gender                   0
admission_type_id        0
dtype: int64

In [128]:
df["max_glu_serum"] = df["max_glu_serum"].fillna("not_measured")
df["A1Cresult"] = df["A1Cresult"].fillna("not_measured")

In [129]:
for col in df.columns:
    print(df[col].value_counts())
    print(df[col].dtype)
    print()

race
Caucasian          76099
AfricanAmerican    19210
?                   2273
Hispanic            2037
Other               1506
Asian                641
Name: count, dtype: int64
str



gender
Female             54708
Male               47055
Unknown/Invalid        3
Name: count, dtype: int64
str

age
[70-80)     26068
[60-70)     22483
[50-60)     17256
[80-90)     17197
[40-50)      9685
[30-40)      3775
[90-100)     2793
[20-30)      1657
[10-20)       691
[0-10)        161
Name: count, dtype: int64
str

weight
?            98569
[75-100)      1336
[50-75)        897
[100-125)      625
[125-150)      145
[25-50)         97
[0-25)          48
[150-175)       35
[175-200)       11
>200             3
Name: count, dtype: int64
str

admission_type_id
1    53990
3    18869
2    18480
6     5291
5     4785
8      320
7       21
4       10
Name: count, dtype: int64
int64

discharge_disposition_id
1     60234
3     13954
6     12902
18     3691
2      2128
22     1993
11     1642
5      1184
25      989
4       815
7       623
23      412
13      399
14      372
28      139
8       108
15       63
24       48
9        21
17       14
16       11
19        8
10        6
27  

In [130]:
df = df[df["gender"] != "Unknown/Invalid"]
df = df[~df["discharge_disposition_id"].isin([11, 13, 14, 19, 20, 21])] # смерть/хоспис
df = df.replace("?", np.nan)

In [131]:
df["admission_type_id"] = df["admission_type_id"].astype(str)
df["discharge_disposition_id"] = df["discharge_disposition_id"].astype(str)
df["admission_source_id"] = df["admission_source_id"].astype(str)

In [132]:
y = (df["readmitted"] == "<30").astype(int)
X = df.drop(columns=["readmitted", "weight", 'examide', 'citoglipton'], errors="ignore")

In [133]:
X.head()

,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,...,troglitazone,tolazamide,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed
0,Caucasian,Female,[0-10),6,25,1,1,NaN,Pediatrics-Endocrinology,41,...,No,No,No,No,No,No,No,No,No,No
1,Caucasian,Female,[10-20),1,1,7,3,NaN,NaN,59,...,No,No,Up,No,No,No,No,No,Ch,Yes
2,AfricanAmerican,Female,[20-30),1,1,7,2,NaN,NaN,11,...,No,No,No,No,No,No,No,No,No,Yes
3,Caucasian,Male,[30-40),1,1,7,2,NaN,NaN,44,...,No,No,Up,No,No,No,No,No,Ch,Yes
4,Caucasian,Male,[40-50),1,1,7,1,NaN,NaN,51,...,No,No,Steady,No,No,No,No,No,Ch,Yes


In [134]:
y.mean()

np.float64(0.1138916851218039)

In [135]:
age_mapping = {
    "[0-10)": 5,
    "[10-20)": 15,
    "[20-30)": 25,
    "[30-40)": 35,
    "[40-50)": 45,
    "[50-60)": 55,
    "[60-70)": 65,
    "[70-80)": 75,
    "[80-90)": 85,
    "[90-100)": 95
}

X["age_mid"] = X["age"].map(age_mapping)

In [136]:
def map_icd9(code):

    if pd.isna(code):
        return 'Missing'

    code = str(code).strip()

    if code.startswith('V'):
        return 'Supplementary_Factors'

    if code.startswith('E'):
        return 'External_Causes'

    try:
        num = float(code)
    except ValueError:
        return 'Unknown'

    if 1 <= num <= 139:
        return 'Infectious_Parasitic'

    elif 140 <= num <= 239:
        return 'Neoplasms'

    elif 240 <= num <= 279:
        return 'Endocrine_Metabolic_Immunity'

    elif 280 <= num <= 289:
        return 'Blood_Diseases'

    elif 290 <= num <= 319:
        return 'Mental_Disorders'

    elif 320 <= num <= 389:
        return 'Nervous_Sense_Organs'

    elif 390 <= num <= 459:
        return 'Circulatory'

    elif 460 <= num <= 519:
        return 'Respiratory'

    elif 520 <= num <= 579:
        return 'Digestive'

    elif 580 <= num <= 629:
        return 'Genitourinary'

    elif 630 <= num <= 679:
        return 'Pregnancy_Childbirth'

    elif 680 <= num <= 709:
        return 'Skin_Subcutaneous'

    elif 710 <= num <= 739:
        return 'Musculoskeletal'

    elif 740 <= num <= 759:
        return 'Congenital_Anomalies'

    elif 760 <= num <= 779:
        return 'Perinatal'

    elif 780 <= num <= 799:
        return 'Symptoms_IllDefined'

    elif 800 <= num <= 999:
        return 'Injury_Poisoning'

    else:
        return 'Unknown'

for col in ["diag_1", "diag_2", "diag_3"]:
    X[col + "_group"] = X[col].apply(map_icd9)

In [137]:
X["prior_visits"] = X["number_outpatient"] + X["number_emergency"] + X["number_inpatient"]

X["had_prior_inpatient"] = (X["number_inpatient"] > 0).astype(int)

X["treatment_intensity"] = X["num_lab_procedures"] + X["num_procedures"] + X["num_medications"]

X["many_medications"] = (X["num_medications"] >= X["num_medications"].quantile(0.75)).astype(int)

X["long_stay"] = (X["time_in_hospital"] >= X["time_in_hospital"].quantile(0.75)).astype(int)

In [138]:
cat_features = X.select_dtypes(include=["str"]).columns.tolist()
num_features = X.select_dtypes(exclude=["str"]).columns.tolist()

In [139]:
for col in cat_features:
    X[col] = X[col].fillna("missing")

for col in num_features:
    X[col] = X[col].fillna(X[col].median())

In [140]:
from sklearn.model_selection import train_test_split

X_train_val, X_test_cal, y_train_val, y_test_cal = train_test_split(
    X,
    y,
    test_size=0.3,
    stratify=y,
    random_state=52
)

X_test, X_cal, y_test, y_cal = train_test_split(
    X_test_cal,
    y_test_cal,
    test_size=0.5,
    stratify=y_test_cal,
    random_state=52
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.2,
    stratify=y_train_val,
    random_state=52
)

In [141]:
from catboost import CatBoostClassifier, Pool

train_pool = Pool(
    X_train,
    y_train,
    cat_features=cat_features
)

valid_pool = Pool(
    X_val,
    y_val,
    cat_features=cat_features
)

cal_pool = Pool(
    X_cal,
    y_cal,
    cat_features=cat_features
)

test_pool = Pool(
    X_test,
    y_test,
    cat_features=cat_features
)

In [173]:
cb_model = CatBoostClassifier(
    iterations = 3000,
    learning_rate = 0.05,
    loss_function = "Logloss",
    eval_metric = "AUC",
    depth = 4,
    l2_leaf_reg = 5,
    auto_class_weights = "SqrtBalanced",
    random_seed = 52,
    early_stopping_rounds = 100,
    verbose = 100,
    allow_writing_files = False
)

cb_model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

0:	test: 0.6251382	best: 0.6251382 (0)	total: 84.7ms	remaining: 4m 13s
100:	test: 0.6735057	best: 0.6735057 (100)	total: 9.04s	remaining: 4m 19s
200:	test: 0.6773277	best: 0.6773407 (198)	total: 18.2s	remaining: 4m 13s
300:	test: 0.6796364	best: 0.6796373 (299)	total: 28.8s	remaining: 4m 17s
400:	test: 0.6809786	best: 0.6809786 (400)	total: 39.3s	remaining: 4m 14s
500:	test: 0.6810739	best: 0.6812233 (420)	total: 50.1s	remaining: 4m 9s
600:	test: 0.6815951	best: 0.6816176 (574)	total: 1m	remaining: 4m 1s
700:	test: 0.6817851	best: 0.6818221 (658)	total: 1m 10s	remaining: 3m 52s
800:	test: 0.6818789	best: 0.6819350 (784)	total: 1m 23s	remaining: 3m 49s
900:	test: 0.6820100	best: 0.6822478 (846)	total: 1m 34s	remaining: 3m 39s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.6822478169
bestIteration = 846

Shrink model to first 847 iterations.


CatBoostClassifier(allow_writing_files=False, auto_class_weights='SqrtBalanced', depth=4, early_stopping_rounds=100, eval_metric='AUC', iterations=3000, l2_leaf_reg=5, learning_rate=0.05, loss_function='Logloss', random_seed=52, verbose=100)

In [174]:
best_iteration = cb_model.get_best_iteration()

final_cb_model = CatBoostClassifier(
    iterations=best_iteration + 1,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    depth = 4,
    l2_leaf_reg = 5,
    auto_class_weights = "SqrtBalanced",
    random_seed=52,
    verbose=100,
    allow_writing_files=False
)

final_train_pool = Pool(X_train_val, y_train_val, cat_features=cat_features)

final_cb_model.fit(final_train_pool)

0:	total: 101ms	remaining: 1m 25s
100:	total: 10.6s	remaining: 1m 18s
200:	total: 23.1s	remaining: 1m 14s
300:	total: 34.4s	remaining: 1m 2s
400:	total: 44.4s	remaining: 49.4s
500:	total: 54.1s	remaining: 37.3s
600:	total: 1m 5s	remaining: 26.9s
700:	total: 1m 16s	remaining: 15.9s
800:	total: 1m 26s	remaining: 4.99s
846:	total: 1m 31s	remaining: 0us


CatBoostClassifier(allow_writing_files=False, auto_class_weights='SqrtBalanced', depth=4, eval_metric='AUC', iterations=847, l2_leaf_reg=5, learning_rate=0.05, loss_function='Logloss', random_seed=52, verbose=100)

In [175]:
def evaluate_proba(y_true, proba):
    print("ROC AUC:", roc_auc_score(y_true, proba))
    print("PR AUC:", average_precision_score(y_true, proba))
    print("Mean predicted probability:", proba.mean())
    print("Real positive rate:", y_true.mean())
    print()


def evaluate_classification(y_true, proba, threshold):
    y_pred = (proba >= threshold).astype(int)
    
    print("Threshold:", threshold)
    print("Precision:", precision_score(y_true, y_pred, zero_division=0))
    print("Recall:", recall_score(y_true, y_pred, zero_division=0))
    print("F1:", f1_score(y_true, y_pred, zero_division=0))
    print()
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print()
    print("Classification report:")
    print(classification_report(y_true, y_pred, zero_division=0))
    print()

In [176]:
test_proba = final_cb_model.predict_proba(test_pool)[:, 1]
cal_proba = final_cb_model.predict_proba(cal_pool)[:, 1]

evaluate_proba(y_test, test_proba)

evaluate_classification(y_test, test_proba, threshold=0.113)

ROC AUC: 0.6784095353687396
PR AUC: 0.24945372348835287
Mean predicted probability: 0.2483941795817419
Real positive rate: 0.11388497416280786

Threshold: 0.113
Precision: 0.11853064711326262
Recall: 0.981143193871538
F1: 0.21150914634146342

Confusion matrix:
[[  822 12382]
 [   32  1665]]

Classification report:
              precision    recall  f1-score   support

           0       0.96      0.06      0.12     13204
           1       0.12      0.98      0.21      1697

    accuracy                           0.17     14901
   macro avg       0.54      0.52      0.16     14901
weighted avg       0.87      0.17      0.13     14901




In [177]:
eps = 1e-6

cat_logit_cal = np.log(np.clip(cal_proba, eps, 1 - eps) / (1 - np.clip(cal_proba, eps, 1 - eps))).reshape(-1, 1)
cat_logit_test = np.log(np.clip(test_proba, eps, 1 - eps) / (1 - np.clip(test_proba, eps, 1 - eps))).reshape(-1, 1)

In [178]:
from sklearn.linear_model import LogisticRegression

cat_calibrator = LogisticRegression()
cat_calibrator.fit(cat_logit_cal, y_cal)
cat_proba_test_calibrated = cat_calibrator.predict_proba(cat_logit_test)[:, 1]

In [179]:
evaluate_proba(y_test, cat_proba_test_calibrated)

evaluate_classification(y_test, cat_proba_test_calibrated, threshold=0.113)

ROC AUC: 0.6784095353687396
PR AUC: 0.24945372348835287
Mean predicted probability: 0.11451830290873996
Real positive rate: 0.11388497416280786

Threshold: 0.113
Precision: 0.18207075384330432
Recall: 0.5792575132586918
F1: 0.27705749718151074

Confusion matrix:
[[8788 4416]
 [ 714  983]]

Classification report:
              precision    recall  f1-score   support

           0       0.92      0.67      0.77     13204
           1       0.18      0.58      0.28      1697

    accuracy                           0.66     14901
   macro avg       0.55      0.62      0.53     14901
weighted avg       0.84      0.66      0.72     14901




In [180]:
final_proba = cat_proba_test_calibrated

threshold_20 = np.quantile(final_proba, 0.8)

evaluate_classification(y_test, final_proba, threshold=threshold_20)

print("Порог, отсекающий топ-20%:", threshold_20)

Threshold: 0.15646103171693454
Precision: 0.23482053002348205
Recall: 0.41249263406010606
F1: 0.2992731936725096

Confusion matrix:
[[10923  2281]
 [  997   700]]

Classification report:
              precision    recall  f1-score   support

           0       0.92      0.83      0.87     13204
           1       0.23      0.41      0.30      1697

    accuracy                           0.78     14901
   macro avg       0.58      0.62      0.58     14901
weighted avg       0.84      0.78      0.80     14901


Порог, отсекающий топ-20%: 0.15646103171693454


In [182]:
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": final_cb_model.get_feature_importance(final_train_pool)
}).sort_values("importance", ascending=False)

feature_importance.head(10)

,feature,importance
4,discharge_disposition_id,18.767747
14,number_inpatient,16.385291
49,had_prior_inpatient,8.805080
15,diag_1,5.041919
48,prior_visits,4.217191
45,diag_1_group,4.149632
11,num_medications,2.924318
8,medical_specialty,2.917078
6,time_in_hospital,2.858742
7,payer_code,2.732767
